**Definitions:**

- **Optimized Write** is a Delta Lake feature that combines many small writes(write to delta table) into one big write for the same partition.
- It happens during the write itself, so fewer small part files get created. which can impact the performance so,

- This is a write-time optimization, performed before data is committed to the Delta table.

- Helps reduce the number of small files being created during ingestion.

**This is done by Compaction:**

- Merges many small files into larger ones (target ~1 GB per file).

- This is a post-write optimization, performed after the data is written to the Delta table.

- Improves query performance by reducing file scan overhead.

- May require VACUUM to clean up obsolete files and free up storage.

**Why is this needed?**
- Let’s take an example:

  -   A company like Flipkart maintains user transaction data in a Delta Table.

    - Suppose:

      - Day 1 → 1,000 transactions = 1,000 part-files

      - Day 2 → 1,500 new transactions = 1,500 new part-files

    - These part-files are stored physically (though we don’t see them directly, they’re tracked in the Delta log).

**Problem:**
- When querying such a table (e.g., SELECT * WHERE salary = 5000), Delta Lake will scan every part-file. 

- This results in:

  - Slower query performance

  - High I/O cost

**Solution: Compaction**

- Compaction merges multiple part-files (say 1,500) into fewer, larger files (e.g., 5).

- It’s typically a periodic task, done weekly/monthly—not daily—to avoid excessive cost.

- Delta Lake aims for ~1 GB per part-file(default). If the data exceeds that, then multiple files are created.

Spark SQL Syntax:

    OPTIMIZE delta.`/path/to/table` ZORDER BY (column1, column2)

PySpark API:

    df.optimize().executeCompaction()
    df.optimize().executeZOrderBy("column1", "column2")

